# Can we predict 30-day readmission?
Target: `readmit_30d`. Data: 11,868 hospital stays from 1,500 people (~8 stays each).

Key design point: **rows are not independent** – the same person appears many times. All validation is therefore grouped by `person_key` so no person is in both train and test; otherwise scores would be inflated. `person_key` itself is never used as a feature.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv')
df.shape, df.isna().sum().sum(), df.duplicated().sum()

## 1. Data overview

In [ ]:
print('Readmission rate:', df.readmit_30d.mean().round(4))
print('People:', df.person_key.nunique(), '| stays/person:', df.groupby('person_key').size().describe()[['min','mean','max']].round(1).to_dict())
df.describe().T

In [ ]:
feats=['visit_number','birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
num=[f for f in feats if f!='sex']
fig,axs=plt.subplots(2,4,figsize=(16,7))
for ax,f in zip(axs.ravel(),num):
    q=pd.qcut(df[f],10,duplicates='drop')
    r=df.groupby(q,observed=True).readmit_30d.agg(['mean','size'])
    ax.plot(range(len(r)),r['mean'],'o-'); ax.axhline(df.readmit_30d.mean(),ls='--',c='grey')
    ax.set_title(f); ax.set_xlabel('decile'); ax.set_ylabel('readmit rate')
axs.ravel()[-1].axis('off'); plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

In [ ]:
print(df[num+['readmit_30d']].corr(method='spearman')['readmit_30d'].drop('readmit_30d').sort_values())
g=df.groupby('person_key').readmit_30d.agg(['sum','size'])
print('Observed variance of person readmit rate:', (g['sum']/g['size']).var().round(4),
      '| expected if independent:', (df.readmit_30d.mean()*(1-df.readmit_30d.mean())/g['size']).mean().round(4))

## 2. Models (grouped cross-validation)
Baselines: prevalence-only dummy and logistic regression; then gradient boosting. Metrics: ROC AUC and PR AUC (average precision; baseline = prevalence ≈ 0.20) since classes are imbalanced, plus Brier score.

In [ ]:
X=df[feats].assign(sex=(df.sex=='M').astype(int)); y=df.readmit_30d; groups=df.person_key
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
models={
 'dummy':DummyClassifier(strategy='prior'),
 'logreg':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'hgb':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED),
}
splits=list(cv.split(X,y,groups))
oof={};rows=[]
for n,m in models.items():
    p=cross_val_predict(m,X,y,cv=splits,method='predict_proba')[:,1]
    oof[n]=p
    fold=[roc_auc_score(y.iloc[te],p[te]) for _,te in splits]
    rows.append(dict(model=n,roc_auc=roc_auc_score(y,p),fold_auc_sd=np.std(fold),pr_auc=average_precision_score(y,p),brier=brier_score_loss(y,p)))
pd.DataFrame(rows).set_index('model').round(4)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n in ['logreg','hgb']:
    f,t,_=roc_curve(y,oof[n]); ax[0].plot(f,t,label=n)
    p,r,_=precision_recall_curve(y,oof[n]); ax[1].plot(r,p,label=n)
    fp,mp=calibration_curve(y,oof[n],n_bins=10,strategy='quantile'); ax[2].plot(mp,fp,'o-',label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set_title('ROC'); ax[1].axhline(y.mean(),c='k',ls='--'); ax[1].set_title('Precision-recall')
ax[2].plot([0,.5],[0,.5],'k--'); ax[2].set_title('Calibration'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('observed')
for a in ax: a.legend()
plt.tight_layout(); plt.show()

## 3. Is the signal real?
Label-permutation null under the same grouped CV, and a person-level bootstrap CI for the logistic-regression AUC.

In [ ]:
rng=np.random.default_rng(SEED)
lr=models['logreg']
null=[]
for i in range(30):
    ys=pd.Series(rng.permutation(y.values),index=y.index)
    p=cross_val_predict(lr,X,ys,cv=splits,method='predict_proba')[:,1]
    null.append(roc_auc_score(ys,p))
print('Null AUC: mean %.3f, max %.3f (30 shuffles)'%(np.mean(null),np.max(null)))
ppl=df.person_key.unique(); idx=df.groupby('person_key').indices
boot=[]
for _ in range(500):
    s=rng.choice(ppl,len(ppl)); ii=np.concatenate([idx[k] for k in s])
    boot.append(roc_auc_score(y.iloc[ii],oof['logreg'][ii]))
print('Logreg AUC 95%% CI (person bootstrap): %.3f-%.3f'%tuple(np.percentile(boot,[2.5,97.5])))

## 4. What drives the prediction?

In [ ]:
lr.fit(X,y)
pd.Series(lr[-1].coef_[0],index=X.columns).sort_values().plot.barh(title='Standardised logistic coefficients (full-data fit)'); plt.show()
imp=np.zeros(X.shape[1])
for tr,te in splits:
    m=models['hgb'].fit(X.iloc[tr],y.iloc[tr])
    imp+=permutation_importance(m,X.iloc[te],y.iloc[te],scoring='roc_auc',n_repeats=5,random_state=SEED).importances_mean/len(splits)
pd.Series(imp,index=X.columns).sort_values().plot.barh(title='HGB permutation importance (AUC drop, held-out)'); plt.show()

## 5. Operating point
What would flagging the highest-risk stays buy? (logistic out-of-fold predictions)

In [ ]:
d=pd.DataFrame({'y':y,'p':oof['logreg']}).sort_values('p',ascending=False)
out=[]
for top in [.05,.10,.20,.30]:
    k=int(len(d)*top); out.append(dict(flag_top=f'{top:.0%}',precision=d.y.iloc[:k].mean(),recall=d.y.iloc[:k].sum()/d.y.sum(),lift=d.y.iloc[:k].mean()/d.y.mean()))
pd.DataFrame(out).round(3)

## 6. Does a person's own history help?
Observed person-level readmission variance is ~3x what independence would imply, so there is a persistent person effect. For a returning patient, the outcomes of their **earlier** stays (lower `visit_number`) are known at admission time. We build leakage-safe features from earlier stays only (the current stay's label is excluded) and keep the grouped CV.

Assumptions to confirm with the data owner: `visit_number` is chronological, and earlier stays' readmission flags are available when the prediction is made. This only applies to people with prior stays in the system; first stays get just the prior.

In [ ]:
h=df.sort_values(['person_key','visit_number']).copy()
gg=h.groupby('person_key').readmit_30d
h['prior_n']=gg.cumcount()
h['prior_readm']=gg.cumsum()-h.readmit_30d          # earlier stays only
h['prior_rate']=(h.prior_readm+0.4)/(h.prior_n+2)   # shrunk toward ~0.2 base rate
h['sex']=(h.sex=='M').astype(int)
Xh,yh,gh=h[feats+['prior_readm','prior_rate']],h.readmit_30d,h.person_key
sp=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED).split(Xh,yh,gh))
res={}
for name,cols in [('stay features only',feats),('+ prior-stay history',feats+['prior_readm','prior_rate'])]:
    for mn in ['logreg','hgb']:
        p=cross_val_predict(models[mn],h[cols],yh,cv=sp,method='predict_proba')[:,1]
        res[(name,mn)]=dict(roc_auc=roc_auc_score(yh,p),pr_auc=average_precision_score(yh,p),brier=brier_score_loss(yh,p))
pd.DataFrame(res).T.round(4)

## 7. Conclusions
* **Yes, but modestly from stay-level data alone**: grouped-CV ROC AUC ~0.65 (logistic regression; person-bootstrap CI ~0.63-0.67; label-shuffled null ~0.50) and PR AUC ~0.32 vs 0.20 prevalence. Gradient boosting does not beat logistic regression, so the signal is mostly simple and monotone.
* Drivers: longer stay, more medications at discharge, higher `lab_a` raise risk. BMI, `lab_b`, age, sex, postcode and visit number carry little or none.
* Flagging the top 10% riskiest stays gives ~40% precision (2x lift) but catches only ~20% of readmissions - fine for prioritising, not for reliably identifying individuals.
* **A person's prior readmission history is much more informative** (AUC ~0.82 in section 6), because readmission clusters within people. This is the most useful lever if prior outcomes are available at prediction time; verify that assumption (chronology of `visit_number`, timing of the label) before relying on it.
* Caveats: only one dataset, no temporal split (visit order could be used for a forward-in-time validation), probabilities from the logistic model are reasonably calibrated but not re-calibrated. Row-wise random CV would overstate performance because of repeated persons - hence the grouped CV.